In [ ]:
import cv2
import mediapipe as mp

# Inicialización de módulos de MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_face_mesh = mp.solutions.face_mesh

# Configuración para dibujar
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

# Captura de la cámara
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ No se pudo acceder a la cámara.")
    exit()

# Inicializa FaceMesh
with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        success, image = cap.read()
        if not success:
            print("⚠️ Ignorando cuadro vacío de la cámara.")
            continue

        # Convierte a RGB para MediaPipe
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        results = face_mesh.process(image_rgb)

        # Volvemos a BGR para mostrar con OpenCV
        image.flags.writeable = True
        image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

        # Dibuja los puntos faciales si hay detecciones
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # Dibuja malla completa
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
                )
                # Dibuja contornos
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
                )
                # Dibuja iris (ojos)
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_IRISES,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_iris_connections_style()
                )

        # Muestra la imagen volteada horizontalmente (efecto espejo)
        cv2.imshow('MediaPipe Face Mesh', cv2.flip(image, 1))

        # Presiona ESC para salir
        if cv2.waitKey(5) & 0xFF == 27:
            break

# Libera recursos
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

: 

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
cap = cv2.VideoCapture(0)

LEFT_EYE = [33, 133]
RIGHT_EYE = [362, 263]
# MOUTH = [78, 308, 14, 13]
MOUTH = [80, 310, 16, 15]

def blur_box(image, points):
    """Aplica desenfoque en el rectángulo que encierra los puntos."""
    x_coords = [p[0] for p in points]
    y_coords = [p[1] for p in points]
    x1, y1 = min(x_coords), min(y_coords)
    x2, y2 = max(x_coords), max(y_coords)

    # Expande un poco el área para cubrir bien la región
    pad_x, pad_y = 10, 10
    x1, y1 = max(x1 - pad_x, 0), max(y1 - pad_y, 0)
    x2, y2 = min(x2 + pad_x, image.shape[1]), min(y2 + pad_y, image.shape[0])

    roi = image[y1:y2, x1:x2]
    if roi.size > 0:
        roi = cv2.GaussianBlur(roi, (55, 55), 30)
        image[y1:y2, x1:x2] = roi
    return image

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        if results.multi_face_landmarks:
            h, w, _ = frame.shape
            for face_landmarks in results.multi_face_landmarks:
                landmarks = [(int(lm.x * w), int(lm.y * h)) for lm in face_landmarks.landmark]

                frame = blur_box(frame, [landmarks[i] for i in LEFT_EYE])
                frame = blur_box(frame, [landmarks[i] for i in RIGHT_EYE])
                frame = blur_box(frame, [landmarks[i] for i in MOUTH])

        cv2.imshow("Censura ojos y boca (rectángulos limpios)", cv2.flip(frame, 1))
        if cv2.waitKey(5) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

: 

In [ ]:
from api.face_api.face_censor import FaceCensor

# Puedes probar diferentes configuraciones
censor = FaceCensor(
    mode="pixelate",  # "blur", "black" o "pixelate"
    expand=15,        # Expande más el área censurada
    pixel_size=2     # Tamaño del pixel si se usa modo pixelate
)
censor.run()

🎥 Cámara iniciada | Modo: PIXELATE | Expand: 15px | Cut: False — Presiona 'ESC' para salir.
✅ Cámara cerrada correctamente.


In [ ]:
from api.face_api.face_censor import FaceCensor

# Puedes probar diferentes configuraciones
censor = FaceCensor(
    mode="black",  # "blur", "black" o "pixelate"
    expand=15        # Expande más el área censurada
)
censor.run()

🎥 Cámara iniciada | Modo: BLACK | Expand: 15px | Cut: False — Presiona 'ESC' para salir.
✅ Cámara cerrada correctamente.


In [ ]:
from api.face_api.face_censor import FaceCensor

# Puedes probar diferentes configuraciones
censor = FaceCensor(
    mode="blur",  # "blur", "black" o "pixelate"
    expand=15        # Expande más el área censurada
)
censor.run()

🎥 Cámara iniciada | Modo: BLUR | Expand: 15px | Cut: False — Presiona 'ESC' para salir.
✅ Cámara cerrada correctamente.


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador (usa los mismos modos que con la cámara)
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5
)

# Procesar una imagen desde archivo
censor.process_image("persona.jpg", output_path="persona_censurada.jpg", show=False)

✅ Resolución válida: 2560x1600
💾 Imagen censurada guardada en: persona_censurada.jpg


array([[[21, 27, 22],
        [20, 26, 21],
        [19, 25, 20],
        ...,
        [19, 25, 20],
        [17, 23, 18],
        [16, 22, 17]],

       [[20, 26, 21],
        [18, 24, 19],
        [18, 24, 19],
        ...,
        [18, 24, 19],
        [17, 23, 18],
        [17, 23, 18]],

       [[18, 24, 19],
        [17, 23, 18],
        [17, 23, 18],
        ...,
        [14, 20, 15],
        [16, 22, 17],
        [18, 24, 19]],

       ...,

       [[35, 56, 48],
        [35, 56, 48],
        [34, 57, 49],
        ...,
        [31, 31, 31],
        [27, 27, 27],
        [22, 22, 22]],

       [[33, 56, 48],
        [33, 56, 48],
        [34, 57, 49],
        ...,
        [35, 35, 35],
        [30, 30, 30],
        [22, 22, 22]],

       [[32, 55, 47],
        [31, 54, 46],
        [31, 54, 46],
        ...,
        [38, 38, 38],
        [32, 32, 32],
        [22, 22, 22]]], dtype=uint8)

In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador (usa los mismos modos que con la cámara)
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5,
    cut=True
)

# Procesar una imagen desde archivo
censor.process_image("persona.jpg", output_path="persona_censurada.jpg", show=False)

✅ Resolución válida: 2560x1600
💾 Imagen censurada guardada en: persona_censurada.jpg


array([[[23, 26, 34],
        [27, 31, 36],
        [28, 32, 37],
        ...,
        [ 9, 17, 24],
        [ 9, 17, 24],
        [11, 19, 26]],

       [[27, 30, 38],
        [32, 36, 41],
        [34, 38, 43],
        ...,
        [ 9, 17, 24],
        [ 9, 17, 24],
        [10, 18, 25]],

       [[31, 35, 40],
        [32, 36, 41],
        [34, 38, 43],
        ...,
        [ 7, 17, 24],
        [ 7, 17, 24],
        [ 7, 17, 24]],

       ...,

       [[31, 33, 41],
        [20, 23, 28],
        [16, 19, 24],
        ...,
        [29, 28, 30],
        [36, 35, 37],
        [28, 27, 29]],

       [[32, 32, 38],
        [25, 25, 31],
        [19, 19, 25],
        ...,
        [29, 28, 32],
        [36, 35, 39],
        [28, 27, 31]],

       [[32, 32, 38],
        [26, 26, 32],
        [21, 21, 27],
        ...,
        [30, 29, 33],
        [37, 36, 40],
        [30, 29, 33]]], dtype=uint8)

In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5
)

# Procesar un video
censor.process_video(
    input_path="persona.mp4",
    output_path="persona_censurada.mp4",
    show=False   # Muestra el video mientras se procesa (puedes poner False para hacerlo más rápido)
)

✅ Resolución válida: 1280x720
🎞️ Procesando video: persona.mp4
💾 Guardando en: persona_censurada.mp4
Progreso: 210/216 frames procesados
✅ Video censurado guardado correctamente en: persona_censurada.mp4


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5,
    cut=True
)

# Procesar un video
censor.process_video(
    input_path="persona.mp4",
    output_path="persona_censurada.mp4",
    show=False   # Muestra el video mientras se procesa (puedes poner False para hacerlo más rápido)
)

✅ Resolución válida: 1280x720
🎞️ Procesando video: persona.mp4
💾 Guardando en: persona_censurada.mp4
Progreso: 210/216 frames procesados
✅ Video censurado guardado correctamente en: persona_censurada.mp4


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador (usa los mismos modos que con la cámara)
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5
)

# Procesar una imagen desde archivo
censor.process_image("persona_baja_calida.jpg", output_path="persona_baja_calida_censurada.jpg", show=False)

⚠️ Advertencia: resolución demasiado baja (170x113). Se requiere al menos 1280x720 para procesar correctamente.
🚫 Imagen descartada por baja resolución.


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador (usa los mismos modos que con la cámara)
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5,
    cut=True
)

# Procesar una imagen desde archivo
censor.process_image("persona_baja_calida.jpg", output_path="persona_baja_calida_censurada.jpg", show=False)

⚠️ Advertencia: resolución demasiado baja (170x113). Se requiere al menos 1280x720 para procesar correctamente.
🚫 Imagen descartada por baja resolución.


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5
)

# Procesar un video
censor.process_video(
    input_path="persona_baja_calidad.mp4",
    output_path="persona_baja_calidad_censurada.mp4",
    show=False   # Muestra el video mientras se procesa (puedes poner False para hacerlo más rápido)
)

❌ No se pudo abrir el video: persona_baja_calidad.mp4


In [ ]:
from api.face_api.face_censor import FaceCensor

# Crear el censurador
censor = FaceCensor(
    mode="pixelate", 
    expand=25, 
    pixel_size=5,
    cut=True
)

# Procesar un video
censor.process_video(
    input_path="persona_baja_calidad.mp4",
    output_path="persona_baja_calidad_censurada.mp4",
    show=False   # Muestra el video mientras se procesa (puedes poner False para hacerlo más rápido)
)

❌ No se pudo abrir el video: persona_baja_calidad.mp4


In [ ]:
from api.face_api.face_censor import FaceCensor

censor = FaceCensor(mode="pixelate", expand=25, pixel_size=5, cut=True)

censor.process_image(
    "https://img1.fonwall.ru/o/lh/cara-delevingne-selebrities-girls-photoshoot.jpeg?route=thumb&h=350",
    output_path="C:/Users/Sheen/Downloads/Tesis/src/12345_userID_20251117_224852/rostro_censurado.jpg",
    show=False
)

🌐 Descargando archivo desde URL: https://img1.fonwall.ru/o/lh/cara-delevingne-selebrities-girls-photoshoot.jpeg?route=thumb&h=350
📥 Archivo descargado temporalmente en: C:\Users\Sheen\AppData\Local\Temp\tmpfhkz7u18.jpeg
✅ Resolución válida: 2308x1526
💾 Imagen censurada guardada en: C:/Users/Sheen/Downloads/Tesis/src/12345_userID_20251117_224852/rostro_censurado.jpg


array([[[100,  77, 129],
        [ 57,  31,  84],
        [ 60,  31,  86],
        ...,
        [162, 200, 235],
        [134, 175, 220],
        [132, 172, 230]],

       [[ 85,  68, 119],
        [ 60,  37,  89],
        [ 66,  40,  93],
        ...,
        [167, 204, 238],
        [129, 172, 215],
        [121, 165, 219]],

       [[ 68,  56, 106],
        [ 44,  25,  76],
        [ 83,  60, 112],
        ...,
        [173, 208, 242],
        [135, 176, 215],
        [121, 161, 213]],

       ...,

       [[ 97, 103, 168],
        [155, 166, 224],
        [ 90, 103, 159],
        ...,
        [167, 195, 226],
        [175, 201, 231],
        [162, 178, 215]],

       [[150, 156, 221],
        [157, 168, 226],
        [ 85,  97, 155],
        ...,
        [171, 199, 230],
        [166, 192, 222],
        [163, 179, 216]],

       [[143, 152, 215],
        [143, 153, 213],
        [ 90, 102, 160],
        ...,
        [174, 202, 233],
        [164, 190, 220],
        [163, 179, 216]]

In [15]:
censor = FaceCensor(mode="pixelate", expand=25, pixel_size=5)

censor.process_video(
    "https://www.pexels.com/es-es/download/video/3761461/",
    output_path="C:/Users/Sheen/Downloads/Tesis/src/12345_userID_20251117_224852/persona_censurada.mp4",
    show=False
)

🌐 Descargando archivo desde URL: https://www.pexels.com/es-es/download/video/3761461/
📥 Archivo descargado temporalmente en: C:\Users\Sheen\AppData\Local\Temp\tmpysxcdgj4.jpg
✅ Resolución válida: 2560x1440
🎞️ Procesando video: https://www.pexels.com/es-es/download/video/3761461/
💾 Guardando en: C:/Users/Sheen/Downloads/Tesis/src/12345_userID_20251117_224852/persona_censurada.mp4
Progreso: 300/301 frames procesados
✅ Video censurado guardado correctamente en: C:/Users/Sheen/Downloads/Tesis/src/12345_userID_20251117_224852/persona_censurada.mp4
